# Voronoi grid generators compared

This notebook builds the same two problems with four Voronoi grid generators
for MODFLOW 6 and compares the grids:

| Tool | Mesher | Cell centres written to DISV | Barriers |
|---|---|---|---|
| **vorflow** | Gmsh, then the Voronoi dual | generators | faces aligned with the line |
| **mf6Voronoi** | point cloud (rings around features, a lattice elsewhere), then Shapely's Voronoi | centroids | no |
| **FloPy** | Triangle, then `VoronoiGrid` | generators | no |
| **VOROGRIDGEN** | Lloyd-relaxed centroidal Voronoi | centroids | no |

It uses the benchmark harness in this folder (`bench/`), so the numbers here are
computed the same way as `python workflow.py` computes them. Plan, fairness rules
and metric definitions are in [docs/benchmark-plan.md](../docs/benchmark-plan.md).

**The shared spec.** Each case gives a background size `h_max`, a target size
`h` per feature and one growth ratio `g`. The size grows linearly with
distance from the nearest feature, `h(x) = min(h_max, h_f + (g - 1) d)`. Each
adapter in `bench/adapters/` translates that spec into the tool's own
settings. Its docstring records the translation, including where a tool
can't express the spec exactly: FloPy steps the size in bands, and
mf6Voronoi's ring multiplier was calibrated to growth 1.2.

**Matched cell count.** Grids are compared at the same number of cells. The
harness scales every length in the spec by one factor and searches for the
factor that brings each tool within 5 % of the case's target count.

**Setup.** Run this in the benchmark environment (`environment.yml` here, or
the `.venv` described in the README), with `mf6` and `triangle` in `.bin/`.
VOROGRIDGEN runs only on Windows. On other platforms the notebook writes its
input files and skips it. The builds and MF6 runs take a minute or two.

In [ ]:
import platform
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.collections import LineCollection, PolyCollection

HERE = Path.cwd()
assert (HERE / "bench").is_dir(), "start Jupyter in tool-benchmark/ (the folder holding bench/)"
sys.path.insert(0, str(HERE))

import workflow
from bench.adapters import TOOLS
from bench.adapters.vorogridgen_adapter import write_inputs
from bench.case import load_case, size_field
from bench.grid import open_rings
from bench.matching import match_count
from bench.metrics import cell_metrics, face_metrics, face_table, summarize
from bench.plots import TOOL_STYLE

WORK_D = workflow.WORK_D / "notebook"
MF6_EXE = workflow.exe_path(workflow.BIN_D, "mf6")
KWARGS = workflow.tool_kwargs(workflow.BIN_D, mf6voronoi_multiplier=None)

available = {
    "vorflow": True,
    "mf6voronoi": True,
    "flopy": workflow.exe_path(workflow.BIN_D, "triangle").exists(),
    "vorogridgen": platform.system() == "Windows" and KWARGS["vorogridgen"]["exe"] is not None,
}
print("tools:", {tool: "ok" if ok else "unavailable" for tool, ok in available.items()})
print("mf6:", "ok" if MF6_EXE.exists() else f"missing ({MF6_EXE})")

Helpers for this notebook. The first builds one case with every available tool
at the matched cell count, and the others draw grids with the benchmark's tool
colours. Where VOROGRIDGEN can't run, `build_all` writes its inputs, and
picks up a grid saved by the CI Windows job if one was merged into `work/`
(see the README).

In [ ]:
def build_all(case):
    # Matched-count grid per tool: {tool: (grid, scale)}.
    target = case.targets[0]
    grids = {}
    for tool in TOOLS:
        if not available[tool]:
            if tool == "vorogridgen":
                ws = WORK_D / case.id / tool
                ws.mkdir(parents=True, exist_ok=True)
                write_inputs(case, 1.0, ws)
                saved = workflow.load_grids(case, workflow.WORK_D, target).get(tool)
                if saved is not None:
                    grids[tool] = (saved, saved.info["scale"])
                    print(f"{tool:12s} loaded from a CI run: {saved.ncpl} cells")
                    continue
            print(f"{tool:12s} skipped")
            continue
        result = match_count(tool, case, target, WORK_D / case.id / tool, KWARGS[tool])
        assert result["status"] == "ok", f"{tool} failed:\n{result['error']}"
        grids[tool] = (result["grid"], result["scale"])
        print(f"{tool:12s} {result['grid'].ncpl} cells (target {target}), "
              f"scale {result['scale']:.3f}, {len(result['history'])} builds, "
              f"matched: {result['matched']}")
    return grids


def cell_max_ortho(grid):
    # Largest orthogonality error over each cell's faces, with the centres the tool writes.
    faces = face_table(grid)
    ortho = face_metrics(grid, faces, "written")["ortho_deg"]
    per_cell = np.zeros(grid.ncpl)
    np.maximum.at(per_cell, faces["ci"], ortho)
    np.maximum.at(per_cell, faces["cj"], ortho)
    return per_cell


def draw_grid(ax, grid, values=None, lw=0.15):
    polys = [grid.vertices[r] for r in open_rings(grid.iverts)]
    if values is None:
        pc = PolyCollection(polys, facecolors="white", edgecolors="k", linewidths=lw)
    else:
        pc = PolyCollection(polys, array=values, cmap="YlGnBu", clim=(0, 10), edgecolors="k", linewidths=lw)
    ax.add_collection(pc)
    ax.set_aspect("equal")
    ax.autoscale_view()
    ax.set_xticks([])
    ax.set_yticks([])
    return pc


def draw_features(ax, case):
    for ring in [case.domain.exterior, *case.domain.interiors]:
        ax.plot(*ring.xy, color="k", lw=1)
    for f in case.features:
        colour = "#e34948" if f.barrier else "#0b0b0b"
        if f.kind == "point":
            ax.plot(*f.geometry.xy, "o", color="#e34948", ms=4, mec="white", mew=0.8)
        elif f.kind == "polygon":
            ax.plot(*f.geometry.exterior.xy, color=colour, lw=1, ls="--")
        else:
            for part in getattr(f.geometry, "geoms", [f.geometry]):
                ax.plot(*part.xy, color=colour, lw=1.8 if f.barrier else 1.2)


def plot_meshes(case, grids, zoom):
    # Top row: whole grid; bottom row: the zoom window (x0, x1, y0, y1).
    fig, axes = plt.subplots(2, len(grids), figsize=(4.2 * len(grids), 8.5), squeeze=False, layout="constrained")
    for col, (tool, (grid, _)) in enumerate(grids.items()):
        for row in (0, 1):
            ax = axes[row, col]
            pc = draw_grid(ax, grid, cell_max_ortho(grid), lw=0.15 if row == 0 else 0.4)
            draw_features(ax, case)
            if row == 1:
                ax.set_xlim(zoom[:2])
                ax.set_ylim(zoom[2:])
        axes[0, col].set_title(f"{tool}: {grid.ncpl} cells", color=TOOL_STYLE[tool]["color"])
    fig.colorbar(pc, ax=axes.ravel().tolist(), shrink=0.6,
                 label="largest orthogonality error of the cell's faces, deg (as MF6 reads it)")
    return fig

## 1. The vorflow demo case

`f4_vorflow_demo` is the geometry of `examples/comprehensive_demo.ipynb`: a
200 × 200 m domain with a square hole, a refined zone (h = 3 m), a sinusoidal
river (2 m), a fault that runs into the hole (2 m, a barrier) and two wells
(1 m). `h_max` is 20 m and the growth ratio 1.2. The target is 3000 cells.

The map shows the spec's target size. Every tool gets the same spec.

In [ ]:
demo = load_case(workflow.CASES_D / "f4_vorflow_demo.yml")
print(demo.description)

x0, y0, x1, y1 = demo.domain.bounds
xx, yy = np.meshgrid(np.linspace(x0, x1, 301), np.linspace(y0, y1, 301))
spec = size_field(demo, np.column_stack([xx.ravel(), yy.ravel()])).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(6, 5.5))
im = ax.pcolormesh(xx, yy, spec, cmap="magma_r", shading="auto")
draw_features(ax, demo)
ax.set_aspect("equal")
fig.colorbar(im, ax=ax, label="spec cell size, m")
ax.set_title("f4_vorflow_demo: target cell size")
plt.show()

In [ ]:
demo_grids = build_all(demo)

### Grids

Cells are coloured by the largest orthogonality error of their faces, computed
the way MODFLOW 6 sees the grid: from the DISV vertices and the centres the tool
writes to `CELL2D`. Without XT3D, MF6 assumes each connection is perpendicular to
its face, so this error measures how far that assumption fails.

- **vorflow and FloPy** write the Voronoi generators. Each face is the
  perpendicular bisector of the line between its two generators, so interior
  faces have (numerically) zero error. Where vorflow's fault splits a cell, the
  piece on the far side gets a mirror generator (the original reflected across
  the fault), so faces next to the barrier stay orthogonal too.
- **mf6Voronoi and VOROGRIDGEN** write cell centroids. A centroid is not a
  generator, so most faces have some error, and it grows where the cell size
  changes quickly.

The bottom row zooms in where the river crosses the fault.

In [ ]:
plot_meshes(demo, demo_grids, zoom=(85, 115, 50, 80))
plt.show()

### Metrics

`summarize` returns the benchmark's scalar metrics for one grid (definitions in
`bench/metrics.py`). This table shows the ones that matter most when choosing a
grid:

- **size / spec**: cell size `sqrt(2A/√3)` over the spec size at the cell centre,
  median over interior cells. 1 means the tool follows the spec.
- **compactness p05**: `4πA/P²` (1 for a circle, 0.91 for a regular hexagon),
  5th percentile. Low values are slivers.
- **ortho p95 / max** and **skew p95**: orthogonality error (degrees) and CVFD face
  skewness over interior faces, using the written centres.
- **disconnected faces**: edges where two cells touch but share no DISV edge
  (a hanging node, or vertices that differ in the last digits). MF6 treats
  them as no-flow.
- **centre offset p95**: |generator − written centre| / √area. It is 0 for
  tools that write generators.
- **mesh s**: wall time of the mesh step alone.

In [ ]:
COLUMNS = {
    "ncpl": "cells",
    "h_ratio_p50": "size / spec",
    "compactness_p05": "compactness p05",
    "ortho_mf6_p95": "ortho p95 (deg)",
    "ortho_mf6_max": "ortho max (deg)",
    "skew_mf6_p95": "skew p95",
    "n_disconnected_faces": "disconnected faces",
    "centre_offset_p95": "centre offset p95",
    "mesh_s": "mesh s",
}


def metrics_table(case, grids):
    rows = {}
    for tool, (grid, scale) in grids.items():
        rows[tool] = {**summarize(grid, case, scale), **grid.timings}
    table = pd.DataFrame(rows).T.reindex(columns=list(COLUMNS)).rename(columns=COLUMNS)
    return table.astype(float).round(3)


metrics_table(demo, demo_grids)

### Following the spec

Each point is one interior cell: its size against the spec size at its centre.
Points on the diagonal follow the spec exactly.

- FloPy's bands show as plateaus, because each band has a single maximum
  triangle area. Its cells are also finer than the spec near the features, so at
  the matched count its spec scale is larger than the others'.
- mf6Voronoi follows the spec through its rings, then jumps to its background
  lattice once the rings end.
- The small outliers are cells next to fixed nodes: vorflow's barrier node
  pairs and FloPy's densified line nodes.

In [ ]:
def plot_size_vs_spec(case, grids):
    fig, axes = plt.subplots(1, len(grids), figsize=(4 * len(grids), 4), sharex=True, sharey=True, squeeze=False,
                             layout="constrained")
    for ax, (tool, (grid, scale)) in zip(axes[0], grids.items()):
        cells = cell_metrics(grid, case, scale)
        inner = ~np.isin(np.arange(grid.ncpl), face_table(grid)["boundary_cell"])
        spec_h = size_field(case, grid.xc, scale)[inner]
        style = TOOL_STYLE[tool]
        ax.loglog(spec_h, cells["h"][inner], style["marker"], color=style["color"], ms=2, alpha=0.4, ls="none")
        lims = [spec_h.min() * 0.7, spec_h.max() * 1.4]
        ax.plot(lims, lims, color="k", lw=1)
        ax.set_title(tool, color=style["color"])
        ax.set_xlabel("spec size at cell centre, m")
    axes[0, 0].set_ylabel("cell size sqrt(2A/sqrt(3)), m")
    return fig


plot_size_vs_spec(demo, demo_grids)
plt.show()

### MODFLOW 6 check: linear flow

`run_models_on` runs the case's MF6 problems (here `linear`: a unit head drop at
30° to the axes, with exact heads as CHD on every boundary cell). A two-point
flux on an orthogonal grid reproduces a linear head field to round-off, so any
error comes from the grid or its centres. Each grid is solved with the centres
the tool writes (`written`) and with the other convention. Tools that write
generators are also run with centroids, and tools that write centroids with
their generators where they expose them. Each is run with XT3D off and on.
The values are area-weighted L2 head errors over the non-CHD cells.

In [ ]:
def mf6_table(case, grids):
    rows = []
    for tool, (grid, _) in grids.items():
        ws = WORK_D / case.id / tool / "mf6"
        rows += workflow.run_models_on(case, grid, case.targets[0], ws, str(MF6_EXE))
    table = pd.DataFrame(rows)
    table["XT3D"] = table["xt3d"].map({False: "off", True: "on"})
    return table.pivot_table(index=["problem", "tool", "centres"], columns="XT3D", values="l2", sort=False)


if MF6_EXE.exists():
    display(mf6_table(demo, demo_grids).style.format("{:.1e}"))

Things to look for:

- With the centres it writes and XT3D off (the default), vorflow reproduces
  the linear field to round-off.
- FloPy's disconnected faces (hanging nodes) act as short no-flow walls. XT3D
  can't fix that, because the connection is missing from the grid.
- mf6Voronoi's error with the centroids it writes is orthogonality error. The
  same grid with its generators is exact, and so is XT3D with the centroids.
- Swapping vorflow's generators for centroids adds the same kind of error, and
  XT3D removes it again.

## 2. A flow barrier at 30°

`v4_barrier` is a 1000 × 600 m rectangle crossed by a straight barrier at 30°
to the axes, with a 10 m target size on the line. The exact solution for flow
normal to the barrier has a jump at the line, set by the barrier's hydraulic
characteristic.

vorflow is the only tool here that aligns cell faces with a line
(`is_barrier=True`). The others get the barrier as a refinement line. The MF6
`barrier` problem then puts an HFB on every connection whose centre-to-centre
segment crosses the line, the way a modeller would. On a grid without aligned
faces, those HFB faces form a zig-zag around the line.

In [ ]:
barrier_case = load_case(workflow.CASES_D / "v4_barrier.yml")
barrier_grids = build_all(barrier_case)

In [ ]:
def hfb_faces(case, grid, feature_id="barrier"):
    # Faces that get an HFB in the MF6 barrier problem: centres on opposite sides of the line.
    line = case.feature(feature_id).geometry
    p0, p1 = np.asarray(line.coords[0]), np.asarray(line.coords[-1])
    normal = np.array([p1[1] - p0[1], p0[0] - p1[0]])
    side = np.sign((grid.xc - p0) @ normal)
    faces = face_table(grid)
    crossing = side[faces["ci"]] != side[faces["cj"]]
    return np.stack([grid.vertices[faces["va"][crossing]], grid.vertices[faces["vb"][crossing]]], axis=1)


zoom = (440, 560, 240, 360)
fig, axes = plt.subplots(1, len(barrier_grids), figsize=(4.2 * len(barrier_grids), 4.6), squeeze=False,
                         layout="constrained")
for ax, (tool, (grid, _)) in zip(axes[0], barrier_grids.items()):
    draw_grid(ax, grid, lw=0.4)
    segments = hfb_faces(barrier_case, grid)
    ax.add_collection(LineCollection(segments, colors="#e34948", linewidths=2.5))
    ax.plot(*barrier_case.feature("barrier").geometry.xy, color="k", lw=0.8, ls="--")
    ax.scatter(*grid.xc.T, s=4, color=TOOL_STYLE[tool]["color"], zorder=3)
    ax.set_xlim(zoom[:2])
    ax.set_ylim(zoom[2:])
    ax.set_title(f"{tool}: {len(segments)} HFB faces", color=TOOL_STYLE[tool]["color"])
fig.suptitle("HFB faces (red) around the barrier (dashed); dots are the written centres")
plt.show()

In [ ]:
metrics_table(barrier_case, barrier_grids)

In [ ]:
if MF6_EXE.exists():
    display(mf6_table(barrier_case, barrier_grids).style.format("{:.1e}"))

In the `barrier` rows, the grids without aligned faces carry the barrier on
the zig-zag. That makes the barrier longer and moves it off the line, which
gives an error well above round-off. XT3D doesn't help, because the error is in
where the HFB sits, not in the connections. On vorflow's grid the HFB faces
lie on the line, and the error is at round-off with the generators it writes.
The `linear` rows are a control: without the barrier, every grid solves
exactly with its generators or with XT3D.

## Where next

- `python workflow.py` runs every case in `cases/`, including convergence
  sweeps (`v2_mms_*`), radial flow to a well (`v3_thiem`) and the calibration of
  mf6Voronoi's ring multiplier (`c0_point_grading`). It writes
  `results/metrics.csv`, `results/mf6_verification.csv` and the figures.
- `docs/benchmark-plan.md` sets out the fairness rules, the metric definitions
  and the tiers of cases.
- To add a case, drop a YAML file in `cases/`. `load_case` reads it, and
  every adapter builds it unchanged.